# L4 短上下文 held-out · 冻结策略比较
在 Colab 选择 **NVIDIA L4** 并运行全部单元格。原 40 篇 held-out 清单已内置，两个任务 × 两个长度组各 10 篇。使用固定版本的 Qwen3-0.6B、LongBench 和原提示词重新采集。每层 B=128，每步 K=64，rank EMA 与归一化 score EMA 都固定 α=0.5。

主指标是相对本轮 LRU 的重复 miss 总数降幅。summary.csv 给出点估计是否严格超过 10%，并另列配对 95% 区间下界是否也超过 10%。ema_increment.csv 直接比较 score EMA 与当前分数，正数表示平滑额外省下的重复 miss。置信区间由任务和长度组内配对重采样整篇文档 10,000 次得到。


In [ ]:
import hashlib, io, json, os, shutil, subprocess, sys, tempfile
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import torch

assert torch.cuda.is_available() and "L4" in torch.cuda.get_device_name().split(), "请选择 NVIDIA L4 GPU"
run_dir = Path(tempfile.mkdtemp(prefix="l4-short-dev-", dir="/content"))
os.chdir(run_dir)
manifest_bytes = b'{\n  "seed": 20261119,\n  "prior_development_ids": [\n    "1d46294ee8fcc0a64828778b04198fcbe4f75841775e1205",\n    "4bd6930b3b65caf89aa127d679a80294f31f9cfa7cdab703",\n    "5b978c8d5792b07ad99da0fc2639c6046051e9c29825ad25",\n    "cbc3f7193f9fa873108c6ccc73a558eb1d2a97b249a1f930",\n    "e6030e255f6b8c70f7b3c9faf0320d4f7729b4eddacf727f"\n  ],\n  "reserved_development_ids": [\n    "gov_report/eaf2acd9c4a757fefc0b0933bd274cccefa900eb9790f6aa",\n    "gov_report/f328905600a19b088f0d4e91dc305b59262a8f2b4b77682e",\n    "gov_report/666d82617c32878d418693bdd0e7f4ff8716b26017869185",\n    "gov_report/051da025ed4174d751c0be47d05cce6cf5d36d4d7e0897bc",\n    "gov_report/79efb6240ef8486e6a6eeb17feafaae41d5cbd53412d244e",\n    "gov_report/7a1cc68b12da91958bd7ebe2c25c69bae3cf95aee04b5e52",\n    "gov_report/0a835927d5d2333bd2525d5f599e81781d951896cc64ad49",\n    "gov_report/f5b4cea3bba6c3989d58b4ea14b70c27bd7507269c704764",\n    "gov_report/ba50360a6669fb0e2fbd41a020483b9bab271b5ce69b5c87",\n    "gov_report/c2a4cfdac819a761d81184d9708cd683900944d37dbc8696",\n    "multifieldqa_en/13b04172d2261a0c9545a0e3670c1c43105ccab41ce1a726",\n    "multifieldqa_en/40a85211345155859f3d2f7268da928f516a65ac6f8c6e08",\n    "multifieldqa_en/67dd297408f3d52f22ede88df0b62644dac86da77f009e71",\n    "multifieldqa_en/7a503a81877d3baca86f8d7179209e4899823433ab3326f3",\n    "multifieldqa_en/048c12357868d908dfb979ef3c5c6cf7f6d85053acf1cac4",\n    "multifieldqa_en/a40877e222497d3ff2efbeb1926e20600f8aac947820063c",\n    "multifieldqa_en/cd0ecda68ad8031330b971fbfdd3794916e815109f004d3b",\n    "multifieldqa_en/4214e20df9d299bae4fb9ba8d1b4792d516812e86960009c",\n    "multifieldqa_en/aeb6cb26b11fc386727a529761d9d233ec7ba8dea9800b0f",\n    "multifieldqa_en/583c9765ccee578fbc4e86090b48ca54163aa00fd5911cfe"\n  ],\n  "model_revision": "c1899de289a04d12100db370d81485cdf75e47ca",\n  "dataset_revision": "5e628be450b7e67fb7ae6e201bd6d8f7056f7672",\n  "gpu": "NVIDIA L4",\n  "dtype": "torch.bfloat16",\n  "page_size": 16,\n  "top_k": 64,\n  "max_new_tokens": 128,\n  "policy": "LRU with full selection rank after equal last-step and hit state",\n  "heldout": [\n    {\n      "request_id": "gov_report/dc4d20da774e559790d83f40bc02e80ae9176495947a625a",\n      "source_sha256": "e7935f21e3dd98e442ba2726e5075a3cd65ac10c3db4fd8c71c1906bb38add0a",\n      "length_band_start": 4096,\n      "prompt_tokens": 6727\n    },\n    {\n      "request_id": "gov_report/b2fe8963e0f2c588e236ef7b7eaafd3334407c176e9db530",\n      "source_sha256": "589ad26f86f42ae441f0d65adf9f17563709f953bd73c3746dec931e153a128f",\n      "length_band_start": 4096,\n      "prompt_tokens": 5725\n    },\n    {\n      "request_id": "gov_report/d9e8583477ebf870fb0c379dc3a75b798a92e5ee882dd93d",\n      "source_sha256": "67f6c31101f7ff179bf940b402b7dee3919bbbfc265f83fc6a1e5c8d6ea6f038",\n      "length_band_start": 4096,\n      "prompt_tokens": 6257\n    },\n    {\n      "request_id": "gov_report/0867ce8275f7a6d527058c76ff165eb872f89416e4acdf0c",\n      "source_sha256": "972ef759a84852bd38c8e9d04f0031c0f9831889dd34f525c5e2557fa714e4bb",\n      "length_band_start": 4096,\n      "prompt_tokens": 4939\n    },\n    {\n      "request_id": "gov_report/42b42abdc193b23ee5bb240e45bd4f5df99e66ea7044fed7",\n      "source_sha256": "a7815fe7d7617dc6229845f58163140afe0a439633dbb9b34692a854b67bf1f8",\n      "length_band_start": 4096,\n      "prompt_tokens": 7921\n    },\n    {\n      "request_id": "gov_report/88e9633a41a602e5b829a63882d3481d3e18112bc0da9cff",\n      "source_sha256": "4609483a680b3a47331e1864f9d54c13512f82d61c304f08d43f62917484e107",\n      "length_band_start": 4096,\n      "prompt_tokens": 6582\n    },\n    {\n      "request_id": "gov_report/f67887ca668cbec0cc6bec7c9f079d3c7fe75962957e0f2d",\n      "source_sha256": "68307d79b359c9de556988dee43c4543898196c2c3de332124429b748e610d75",\n      "length_band_start": 4096,\n      "prompt_tokens": 4851\n    },\n    {\n      "request_id": "gov_report/a12d1df7a84ab7478fc861b8b0221e9b0dd8d0e6a96d18fc",\n      "source_sha256": "fa2b00174fbcbed26d294a83ec6bacee3e083ba2c145e9cab25dc39438bdf456",\n      "length_band_start": 4096,\n      "prompt_tokens": 7241\n    },\n    {\n      "request_id": "gov_report/99f4c9b3ae17f0f11db1ec1965778c695a021407717a969a",\n      "source_sha256": "c141ef8aa6cb759ebdbd1bb917a617b3eb17a5dec27385468063c7a34195d794",\n      "length_band_start": 4096,\n      "prompt_tokens": 7695\n    },\n    {\n      "request_id": "gov_report/282fcfa4d5fc83fc9149c72060bd90ca36a80b173dacc48f",\n      "source_sha256": "f92301b9dbf8ade3ad867875345bd550db903e23428fd7972487bb2a4dca27cc",\n      "length_band_start": 4096,\n      "prompt_tokens": 4906\n    },\n    {\n      "request_id": "gov_report/8be9961b3148717dd324148abf0ab947fb11adb0b9ca3fc6",\n      "source_sha256": "459596a8f4ddfb27a33470f1fd3b07450e188de5f844d3dda5572ec5299c5d86",\n      "length_band_start": 8192,\n      "prompt_tokens": 14073\n    },\n    {\n      "request_id": "gov_report/7decb8feac7c313ba51e5813869dc0723e029c13c3a6aed1",\n      "source_sha256": "4a910815cfc24adcc61fd7b85f799f9fca7c06f8ebb438ccd46e33410751a251",\n      "length_band_start": 8192,\n      "prompt_tokens": 10555\n    },\n    {\n      "request_id": "gov_report/dc41fbf74e1c82b1902f266a75ae9e43a1605b4879fabf18",\n      "source_sha256": "4480743341dcc527e33ec7cc23c1d10b159b6a0a3fb945af9919037e10a5b5ff",\n      "length_band_start": 8192,\n      "prompt_tokens": 15018\n    },\n    {\n      "request_id": "gov_report/d83f991dcb7c0f3caf4d1f675d4c46e1345881cc66d25fff",\n      "source_sha256": "214e4b884909bac421d944d22d2be8b95bc7f97bcd7e9dd62cf24ee796c510a8",\n      "length_band_start": 8192,\n      "prompt_tokens": 11782\n    },\n    {\n      "request_id": "gov_report/4bdece48fd21b6df53b6716155d6466eb3573a4390e175d2",\n      "source_sha256": "565d66080224d68ac7b64106deb896f26d2f3ee66b8d7bf0694bc6d092477168",\n      "length_band_start": 8192,\n      "prompt_tokens": 11051\n    },\n    {\n      "request_id": "gov_report/155a5ed26a157dfdae3b46f8f627df30831753dab59c6bdf",\n      "source_sha256": "c343f3ac0ec32d0cc39acc1cdd81814f8c9837f9d3f09ef9c52eb82d008d4c86",\n      "length_band_start": 8192,\n      "prompt_tokens": 8208\n    },\n    {\n      "request_id": "gov_report/00cee8bad9ec18d013af6d0e731f1bc7af22da0d06bf6d00",\n      "source_sha256": "d1187333c91ddb45df26f161f06561c2a2437ba09e8693d20cf0a64b7ddbe6c7",\n      "length_band_start": 8192,\n      "prompt_tokens": 8913\n    },\n    {\n      "request_id": "gov_report/610a8939f0468afb49281c0bfd81d3dbf0dd43a787207cd6",\n      "source_sha256": "9158fb68a4599edef68c17c501b6f7ff38e0f7e96b4f60c81b295ee7e3df8f85",\n      "length_band_start": 8192,\n      "prompt_tokens": 14377\n    },\n    {\n      "request_id": "gov_report/52c18fcac6f70755770f9bd45f030388a448a9ad0f9b64be",\n      "source_sha256": "b3dc269730e8ab26294fde43405c305fcf4102607a18858beea1214daa55f1d3",\n      "length_band_start": 8192,\n      "prompt_tokens": 10714\n    },\n    {\n      "request_id": "gov_report/b9daebf4ce1384e0deff985966ebdb1db056ff7c5808d497",\n      "source_sha256": "6839db447ef4cd1bade7b436959fe7d4e6f115fe072bf51bd0f04057c5d1eb1f",\n      "length_band_start": 8192,\n      "prompt_tokens": 13478\n    },\n    {\n      "request_id": "multifieldqa_en/edbbdb9727c3a51310d24895d08c5a90673cb9d514770878",\n      "source_sha256": "c58b4831f2d6fec6a201fc17475a3d799f513999e9d7bea6ae119c5ca5dcd2d4",\n      "length_band_start": 4096,\n      "prompt_tokens": 5181\n    },\n    {\n      "request_id": "multifieldqa_en/37dfd65b9a8316b9569875b8bcfc5a8bd65e044a88b2e894",\n      "source_sha256": "adf2de189dbb844c8b07814b912e5bcfdf7d54d3327baa15571c9ba1ae663a9e",\n      "length_band_start": 4096,\n      "prompt_tokens": 7089\n    },\n    {\n      "request_id": "multifieldqa_en/dffb383ee06d8413ac40e2c1ec7bde5548ef07fc8f35ad2f",\n      "source_sha256": "5b3a8470a4575f1eb1a4f7098d8c8a2d53abd1d9971cc70b6d998aec7bfb7fb8",\n      "length_band_start": 4096,\n      "prompt_tokens": 7920\n    },\n    {\n      "request_id": "multifieldqa_en/238c4efe738cecd8346abfdc57707996aef30f9b43d1a577",\n      "source_sha256": "9e76bd456fedb7d58a9f7be828ee79b835e97b224068543c52215bcc8f22237a",\n      "length_band_start": 4096,\n      "prompt_tokens": 4339\n    },\n    {\n      "request_id": "multifieldqa_en/ac42a5a54b029a05be6e6e13d9001a81761d7b40ad1edfe2",\n      "source_sha256": "020593296ce836b33f4d766ba5508025a42beebc334eeab9c100ce5f4fbd0322",\n      "length_band_start": 4096,\n      "prompt_tokens": 5117\n    },\n    {\n      "request_id": "multifieldqa_en/5a18017502d827b72fc74d91f13f6f14193b4964c3420e47",\n      "source_sha256": "db2210e07439108d62b7be59c5c0a7935fc8a1e538b6c761abbd2feb680f81f7",\n      "length_band_start": 4096,\n      "prompt_tokens": 6534\n    },\n    {\n      "request_id": "multifieldqa_en/8fbf0a6531d9250e6bcda0c7ba456441f6d4073bf08de826",\n      "source_sha256": "5b15814d5cd024e28a48f07e49033506df3f2134f7dcd819ada8897141bae2e1",\n      "length_band_start": 4096,\n      "prompt_tokens": 7698\n    },\n    {\n      "request_id": "multifieldqa_en/c6eba7d0323b89bbdb0fbad233543848add08dcbfcbd875b",\n      "source_sha256": "08581ae8fdcb6b4b2d4944d99f23fc11f7f58bdd1d860e3307af164ee4c67ce5",\n      "length_band_start": 4096,\n      "prompt_tokens": 4529\n    },\n    {\n      "request_id": "multifieldqa_en/120cb783c796fbedbc76f04cf9be3318e54a63cd642c4401",\n      "source_sha256": "bfcb63a48eb8c475bdbd9f3de0da778ce767de1ee64d392efe74e1fdf98d647d",\n      "length_band_start": 4096,\n      "prompt_tokens": 7873\n    },\n    {\n      "request_id": "multifieldqa_en/462011b6f9beb976aaf7b38082bcf7d70a91c3c74d2c6e95",\n      "source_sha256": "e2eb568c84175b9d893ac9022e0170544c82ddbea2392c2cd6bdea95ffbb69dc",\n      "length_band_start": 4096,\n      "prompt_tokens": 4926\n    },\n    {\n      "request_id": "multifieldqa_en/0da7747b164232cac911a34f6d0421028b27db0ce70e26a7",\n      "source_sha256": "a5e4c17962854994fb4ff9324eeeda6a98659aa6bd31beec7c46266e6f9b7111",\n      "length_band_start": 8192,\n      "prompt_tokens": 15601\n    },\n    {\n      "request_id": "multifieldqa_en/e84f6371f23c95f766248ac4e52b781f325f52f09c6eb9ae",\n      "source_sha256": "79ba86dd5d313a6d274c5a55458903569612518b7406560c0b9727b0eadd335d",\n      "length_band_start": 8192,\n      "prompt_tokens": 8551\n    },\n    {\n      "request_id": "multifieldqa_en/e4725ed7b6df9933d5336f79d5c9d1d803a3e475831f1017",\n      "source_sha256": "32d09d8191cdbfda651e4c198b5066fce8def88a08df55a0e43b377ee91110fe",\n      "length_band_start": 8192,\n      "prompt_tokens": 9659\n    },\n    {\n      "request_id": "multifieldqa_en/6ee1971ac8c7c0ee4a5add9ec201557d28d3c2f66b176fb4",\n      "source_sha256": "4b3ca122f5b124ccfcf81315e9346edabc1e20fe4704d3884e61ea00169e1bda",\n      "length_band_start": 8192,\n      "prompt_tokens": 9495\n    },\n    {\n      "request_id": "multifieldqa_en/04b588ff2dea15f4a9c4fdbaabc55aaad1ba3446114d6741",\n      "source_sha256": "2a7a16e86a6ffebb116834d3b9344ad2a9f9eae47cc072a2e2566bbb6d239d88",\n      "length_band_start": 8192,\n      "prompt_tokens": 9598\n    },\n    {\n      "request_id": "multifieldqa_en/135b82aecda3c83cfd73d1447e8f92a8a44de50ffdb09444",\n      "source_sha256": "e45e29c7d21b8bdedf489f16b4a8d834333f021e3181190ad37e9df517f2ae49",\n      "length_band_start": 8192,\n      "prompt_tokens": 9184\n    },\n    {\n      "request_id": "multifieldqa_en/18ef34b54d2ddc134e1be7cae3d6101432465011d016c77a",\n      "source_sha256": "26f777fd3a451da525a96d5bd7935fb1620158148e2ba1fe5cb8994e7bdfb19e",\n      "length_band_start": 8192,\n      "prompt_tokens": 10148\n    },\n    {\n      "request_id": "multifieldqa_en/a8da0d2585bb54d571a2ab4a539b61eef56b5e271692be42",\n      "source_sha256": "96a9735499432f8f40eef8b88586b71a6588dbc3bb5b5ba04826224403324c4a",\n      "length_band_start": 8192,\n      "prompt_tokens": 8681\n    },\n    {\n      "request_id": "multifieldqa_en/2f73cf12085e3fb879c775fc4851b72092a060cad5a927c6",\n      "source_sha256": "92aac287a57cb782ae761c3dd64903f6f164055e50973c310af8216d60c7b589",\n      "length_band_start": 8192,\n      "prompt_tokens": 12020\n    },\n    {\n      "request_id": "multifieldqa_en/917f18543035ee1b9161cded1d3352531dbf3b249b5f0a18",\n      "source_sha256": "75498c823026b5c9b95cfeb27a8d22d3d3d755336fb233d689eb24f42d7b2465",\n      "length_band_start": 8192,\n      "prompt_tokens": 10813\n    }\n  ]\n}'
assert hashlib.sha256(manifest_bytes).hexdigest()=='55c8937af641d17cebc42ca4ddfd91a910ba3b3cd78b264053e76fab47b98d3d'
Path("frozen_heldout_manifest.json").write_bytes(manifest_bytes)
print(torch.cuda.get_device_name(), run_dir)


### 1 · 写入原采集流程与比较程序
沿用原始的分段 prefill、贪心解码、Quest 计算和稳定排序。补录每一步所有已完成历史页在选 top-64 之前的 FP32 分数。

In [ ]:
%%writefile capture.py
import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
import importlib
import subprocess
import sys
from pathlib import Path
from tempfile import mkdtemp

# Use Colab's Torch, NumPy and Matplotlib. Keep the model-specific pins private
# to this notebook so pip leaves Colab's Gradio and Diffusers dependencies alone.
model_packages = {
    "transformers": "4.53.3",
    "huggingface-hub": "0.36.2",
    "tokenizers": "0.21.4",
    "safetensors": "0.8.0",
}
if "model_package_dir" not in globals():
    loaded = [name for name in model_packages if name.replace("-", "_") in sys.modules]
    if loaded:
        raise RuntimeError(
            f"Already imported {', '.join(loaded)}. Restart the Colab session "
            "and run this setup cell first."
        )
    package_dir = Path(mkdtemp(prefix="rank-kv-packages-"))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
         "--target", str(package_dir),
         *(f"{name}=={version}" for name, version in model_packages.items())],
        check=True,
    )
    sys.path.insert(0, str(package_dir))
    importlib.invalidate_caches()
    model_package_dir = package_dir

import huggingface_hub
import transformers
from transformers import AttentionInterface, Qwen3ForCausalLM

if (transformers.__version__ != model_packages["transformers"]
        or huggingface_hub.__version__ != model_packages["huggingface-hub"]):
    raise RuntimeError("Unexpected library versions. Restart the session and run setup first.")
print(f"Setup ready: Transformers {transformers.__version__}, Hugging Face Hub {huggingface_hub.__version__}")

import json
import torch
from torch.nn import functional as F
from transformers import AttentionInterface


class QuestPages:
    """A reference decode backend for one unpadded request. All KV remains on the device."""

    def __init__(self, page_size, top_k):
        self.page_size = page_size
        self.top_k = top_k
        self.bounds = {}
        self.pending = {}
        self.page_scores = {}

    def __call__(self, module, query, key, value, attention_mask, scaling, **kwargs):
        if query.shape[0] != 1 or query.shape[2] != 1:
            raise ValueError("QuestPages expects one request and one decode query")
        layer = module.layer_idx
        kv_heads, length, dim = key.shape[1:]
        # The page containing the query is pinned, including its last token.
        history_pages = (length - 1) // self.page_size
        past = self.bounds.get(layer)
        cached = 0 if past is None else past[0].shape[1]
        if history_pages > cached:
            block = key[0, :, cached*self.page_size:history_pages*self.page_size]
            block = block.reshape(kv_heads, history_pages - cached, self.page_size, dim)
            low, high = block.amin(dim=2).float(), block.amax(dim=2).float()
            if past is not None:
                low = torch.cat((past[0], low), dim=1)
                high = torch.cat((past[1], high), dim=1)
            self.bounds[layer] = low, high
        groups = query.shape[1] // kv_heads
        if history_pages:
            low, high = self.bounds[layer]
            q = query[0, :, 0].reshape(kv_heads, groups, dim).float()
            upper = (torch.einsum("hgd,hpd->hgp", q.clamp_min(0), high)
                     + torch.einsum("hgd,hpd->hgp", q.clamp_max(0), low))
            scores = upper.mean(dim=(0, 1))
            # Page IDs start in ascending order, so stable ties prefer smaller IDs.
            ranked = scores.argsort(descending=True, stable=True)[:self.top_k]
        else:
            ranked = torch.empty(0, dtype=torch.long, device=key.device)
        self.pending[layer] = ranked
        self.page_scores[layer] = scores if history_pages else key.new_empty(0, dtype=torch.float32)
        offsets = torch.arange(self.page_size, device=key.device)
        selected = (ranked.sort().values[:, None] * self.page_size + offsets).flatten()
        current = torch.arange(history_pages*self.page_size, length, device=key.device)
        positions = torch.cat((selected, current))
        sparse_k = key.index_select(2, positions).repeat_interleave(groups, dim=1)
        sparse_v = value.index_select(2, positions).repeat_interleave(groups, dim=1)
        output = F.scaled_dot_product_attention(query, sparse_k, sparse_v,
                                                dropout_p=0.0, is_causal=False, scale=scaling)
        return output.transpose(1, 2).contiguous(), None


@torch.inference_mode()
def capture(model, prompt_ids, request_id, *, page_size, top_k, max_new_tokens, eos_ids):
    if prompt_ids.ndim != 2 or prompt_ids.shape[0] != 1 or prompt_ids.shape[1] < 1:
        raise ValueError("supply one unpadded prompt")
    if max_new_tokens < 2 or page_size < 1 or top_k < 1:
        raise ValueError("invalid token or page budget")
    cfg = model.config
    backend = QuestPages(page_size, top_k)
    original = cfg._attn_implementation
    AttentionInterface.register("rank_kv_quest", backend)
    rows, generated, score_steps = [], [], []
    try:
        cfg._attn_implementation = "sdpa"
        cache = None
        # Chunk queries to bound the temporary attention matrix on 16-GB GPUs.
        for chunk in prompt_ids.split(512, dim=1):
            output = model(input_ids=chunk, past_key_values=cache, use_cache=True, logits_to_keep=1)
            cache = output.past_key_values
        token = output.logits[:, -1].argmax(dim=-1, keepdim=True)
        generated.append(token.item())
        cfg._attn_implementation = "rank_kv_quest"
        for step in range(max_new_tokens - 1):
            if generated[-1] in eos_ids:
                break
            backend.pending.clear()
            output = model(input_ids=token, past_key_values=cache, use_cache=True, logits_to_keep=1)
            cache = output.past_key_values
            if set(backend.pending) != set(range(cfg.num_hidden_layers)):
                raise RuntimeError("incomplete layer trace")
            rankings = torch.stack([backend.pending[i] for i in range(cfg.num_hidden_layers)]).tolist()
            score_steps.append(torch.stack([backend.page_scores[i] for i in range(cfg.num_hidden_layers)]).cpu().numpy())
            expected = min(top_k, (prompt_ids.shape[1] + step) // page_size)
            for layer, ids in enumerate(rankings):
                if len(ids) != expected:
                    raise RuntimeError("incorrect number of completed pages")
                rows.append(dict(request_id=request_id, layer_id=layer, decode_step=step,
                                 ranked_page_ids=ids))
            token = output.logits[:, -1].argmax(dim=-1, keepdim=True)
            generated.append(token.item())
    finally:
        cfg._attn_implementation = original
        # The registered callable outlives this call. Release its request-specific tensors.
        backend.page_scores.clear()
        backend.bounds.clear()
        backend.pending.clear()
    return rows, generated, score_steps

from hashlib import sha256
from pathlib import Path
from zipfile import ZipFile
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_REV = "c1899de289a04d12100db370d81485cdf75e47ca"
DATA_ID = "zai-org/LongBench"
DATA_REV = "5e628be450b7e67fb7ae6e201bd6d8f7056f7672"
PROMPT_TOKENS = 4096
PAGE_SIZE, TOP_K, MAX_NEW = 16, 64, 128
BUDGETS = [64, 128, 256]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV, token=False)
archive = hf_hub_download(DATA_ID, "data.zip", repo_type="dataset", revision=DATA_REV, token=False)
prefix = tokenizer.encode("Government report excerpt:\n\n", add_special_tokens=False)
suffix = tokenizer.encode("\n\nWrite a one-paragraph summary of this report.\nSummary:", add_special_tokens=False)

import transformers
from transformers import AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("In Colab choose Runtime → Change runtime type → a GPU, then Run all.")
device = torch.device("cuda")
assert torch.cuda.get_device_capability()[0] >= 8, "请选择原采集所用的 A100 或其他支持原生 BF16 的 GPU"
dtype = torch.bfloat16
torch.backends.cuda.matmul.allow_tf32 = False
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REV, torch_dtype=dtype, attn_implementation="sdpa", token=False).to(device).eval()
eos = model.generation_config.eos_token_id
eos_ids = {eos} if isinstance(eos, int) else set(eos or [])

import numpy as np
import platform
from validation import as_array, check_scores, write_csv
out=Path("result")
store=out/"scores"
store.mkdir(parents=True,exist_ok=True)
with ZipFile(archive) as dataset:
    m=json.loads(Path("frozen_heldout_manifest.json").read_text())
    docs=m["heldout"]
    assert len(docs)==len({x["request_id"] for x in docs})==40
    heldout_ids={d["request_id"] for d in docs}
    assert not heldout_ids.intersection(m["reserved_development_ids"])
    assert not {rid.split("/")[1] for rid in heldout_ids}.intersection(m["prior_development_ids"])
    assert len({d["source_sha256"] for d in docs})==40
    assert (MODEL_REV,DATA_REV,PAGE_SIZE,TOP_K,MAX_NEW)==(m["model_revision"],m["dataset_revision"],m["page_size"],m["top_k"],m["max_new_tokens"])
    assert model.config.num_hidden_layers==28
    (out/"original_manifest.json").write_text(json.dumps(m,indent=2))
    (out/"environment.json").write_text(json.dumps(dict(gpu=torch.cuda.get_device_name(),torch=torch.__version__,
        cuda=torch.version.cuda,transformers=transformers.__version__,model_packages=model_packages,python=platform.python_version(),
        numpy=np.__version__,score_dtype="float32",tf32_matmul=torch.backends.cuda.matmul.allow_tf32,
        B=128,K=64,alpha=0.5,seed=m["seed"],model_revision=MODEL_REV,dataset_revision=DATA_REV),indent=2))
    lookup={}
    for task in ("gov_report","multifieldqa_en"):
        filename=next(n for n in dataset.namelist() if Path(n).name==task+".jsonl")
        for line in dataset.read(filename).splitlines():
            d=json.loads(line)
            if task+"/"+d["_id"] in heldout_ids: lookup[task+"/"+d["_id"]]=d
    audits=[]
    for i,doc in enumerate(docs,1):
        rid=doc["request_id"]
        d=lookup[rid]; gov=rid.startswith("gov_report/")
        assert sha256(d["context"].encode()).hexdigest()==doc["source_sha256"]
        head=prefix if gov else tokenizer.encode("Document:\n",add_special_tokens=False)
        tail=suffix if gov else tokenizer.encode("\nQuestion: "+d["input"]+"\nAnswer:",add_special_tokens=False)
        tokens=head+tokenizer.encode(d["context"],add_special_tokens=False)+tail
        assert len(tokens)==doc["prompt_tokens"]
        records,generated,steps=capture(model,torch.tensor([tokens],device=device),rid,page_size=16,top_k=64,max_new_tokens=128,eos_ids=eos_ids)
        current=as_array(records,rid)
        check_scores(current,steps,len(tokens))
        assert len(generated)==len(steps)+1
        report=dict(doc=rid,steps=len(steps),layers=28,ranked_rows_checked=len(steps)*28,passed=True)
        values=np.full((len(steps),28,steps[-1].shape[1]),np.nan,dtype=np.float32)
        for t,v in enumerate(steps): values[t,:,:v.shape[1]]=v
        np.savez_compressed(store/f"{i:02d}.npz",rid=rid,ranked=current.astype(np.int16),scores=values,
                            generated=np.array(generated),prompt_tokens=len(tokens))
        del values
        del steps
        audits.append(report)
        write_csv(out/"l4_trace_checks.csv",audits)
        print(f"{i}/40 {rid}: {report['ranked_rows_checked']} ordered top-64 rows verified",flush=True)
assert len(audits)==40


In [ ]:
%%writefile sim.py
import heapq
import numpy as np

class Policy:
    """Base: LRU bookkeeping. evict() returns n victims from candidates (unselected residents)."""
    needs_scores = False

    def __init__(self, K, **kw):
        self.K = K
        self.last = {}
        self.hit = {}
        self.params = kw

    def begin(self, t, ranked, srow):
        self.t, self.ranked, self.srow = t, ranked, srow

    def evict(self, cands, n):
        return heapq.nsmallest(n, cands, key=self.key)

    def key(self, p):
        return self.last[p], self.hit[p], -p

    def dropped(self, p):
        self.last.pop(p, None)
        self.hit.pop(p, None)

    def admit(self, hits, misses):
        for p in self.ranked:
            self.last[p] = self.t
            self.hit[p] = p in hits

class LRU(Policy):
    """tie: how pages last selected at the same step are ordered (key ascending = evicted first)
       "miss_first" (default, as replay.py): among equals, pages that were misses leave first
       "hit_first": pages that were hits leave first (newly loaded pages are protected)
       "id": larger page ID leaves first"""
    def __init__(self, K, tie="miss_first", **kw):
        super().__init__(K, **kw)
        if tie not in ("miss_first", "hit_first", "id"):
            raise ValueError("bad tie")
        self.tie = tie

    def key(self, p):
        if self.tie == "miss_first":
            return self.last[p], self.hit[p], -p
        if self.tie == "hit_first":
            return self.last[p], not self.hit[p], -p
        return self.last[p], -p

def _z(srow, ranked, p):
    """Score of p relative to the K-th selected score, in units of (top1 - K-th)."""
    kth, top = srow[ranked[-1]], srow[ranked[0]]
    return (srow[p] - kth) / (top - kth + 1e-6)

class ScoreCurrent(Policy):
    """Evict the lowest current selector score (candidates nearest to re-selection stay)."""
    needs_scores = True

    def key(self, p):
        return self.srow[p], self.last[p], self.hit[p], -p

class ScoreEMA(Policy):
    """Evict the lowest exponential moving average of a per-step page signal (one float per resident).

    signal: "z"    score relative to the K-th selected score, in units of (top1 - K-th)
            "raw"  the raw selector score
            "rank" rank-only information (schema v1): (K - rank + 1) / K if selected, else 0
    mode:   "all"       every resident is updated every step (needs every resident's current score)
            "selected"  a resident is updated only on steps where it is selected
    """
    needs_scores = True

    def __init__(self, K, alpha=0.5, mode="all", signal="z", **kw):
        super().__init__(K, **kw)
        if mode not in ("all", "selected") or signal not in ("z", "raw", "rank"):
            raise ValueError("bad mode or signal")
        self.alpha, self.mode, self.signal = alpha, mode, signal
        self.ema = {}

    def value(self, p):
        if self.signal == "z":
            return _z(self.srow, self.ranked, p)
        if self.signal == "raw":
            return self.srow[p]
        r = self.rank_of.get(p)
        return 0.0 if r is None else (self.K - r + 1) / self.K

    def begin(self, t, ranked, srow):
        super().begin(t, ranked, srow)
        self.rank_of = {p: r for r, p in enumerate(ranked, 1)}
        a = self.alpha
        pages = self.ema if self.mode == "all" else [p for p in ranked if p in self.ema]
        for p in pages:
            self.ema[p] = a * self.ema[p] + (1 - a) * self.value(p)

    def key(self, p):
        return self.ema[p], self.last[p], self.hit[p], -p

    def dropped(self, p):
        super().dropped(p)
        del self.ema[p]

    def admit(self, hits, misses):
        super().admit(hits, misses)
        for p in misses:
            self.ema[p] = self.value(p)

def simulate(ranked, scores, B, name, *, on_evict=None, **params):
    """Replay one layer stream. ranked: [T, K] ints; scores: [T, P] or None.
    Returns dict(accesses, misses, compulsory). on_evict(t, victims, policy) is optional."""
    rows = [tuple(int(p) for p in r if p >= 0) for r in np.asarray(ranked)]
    K = max(len(r) for r in rows)
    if B < K:
        raise ValueError("budget below K")
    cls = POLICIES[name]
    pol = cls(K, **params)
    if cls.needs_scores and scores is None:
        raise ValueError(f"{name} needs scores")
    res, seen = set(), set()
    acc = miss = 0
    for t, row in enumerate(rows):
        srow = scores[t].tolist() if cls.needs_scores else None
        pol.begin(t, row, srow)
        sel = set(row)
        hits = sel & res
        misses = sel - hits
        n = len(res) + len(misses) - B
        if n > 0:
            victims = pol.evict(res - sel, n)
            if on_evict is not None:
                on_evict(t, victims, pol)
            for v in victims:
                res.discard(v)
                pol.dropped(v)
        res |= sel
        pol.admit(hits, misses)
        assert len(res) <= B
        acc += len(sel)
        miss += len(misses)
        seen |= sel
    return dict(accesses=acc, misses=miss, compulsory=len(seen))

POLICIES = {"lru":LRU,"score":ScoreCurrent,"ema":ScoreEMA}

In [ ]:
%%writefile validation.py
import csv, json
from pathlib import Path
import numpy as np

def as_array(rows,rid):
    if not rows or len(rows)%28: raise ValueError(f"Invalid trace length: {rid}")
    a = np.full((len(rows)//28,28,64),-1,dtype=np.int32)
    for row in rows:
        t,l=row["decode_step"],row["layer_id"]
        p=row["ranked_page_ids"]
        if (row["request_id"]!=rid or not 0<=t<len(a) or not 0<=l<28 or len(p)!=64 or
            min(p)<0 or len(set(p))!=64 or a[t,l,0]!=-1): raise ValueError(f"Invalid trace row: {rid}/{t}/{l}")
        a[t,l]=p
    if (a<0).any(): raise ValueError("Incomplete trace")
    return a

def check_scores(current,scores,tokens):
    assert len(scores)==len(current) and len(scores)>0
    for t,values in enumerate(scores):
        assert values.dtype==np.float32 and values.shape==(28,(tokens+t)//16) and np.isfinite(values).all()
        order=np.argsort(-values,axis=1,kind="stable")[:,:64]
        if not np.array_equal(order,current[t]):
            raise ValueError(f"L4 FP32 scores and observed top-64 disagree at step {t}")

def write_csv(path, rows):
    with Path(path).open("w",newline="") as f:
        w=csv.DictWriter(f,rows[0].keys()); w.writeheader(); w.writerows(rows)

In [ ]:
%%writefile compare.py
import csv,json,math
from collections import defaultdict
from pathlib import Path
import numpy as np
from sim import simulate
from validation import write_csv

POLICIES={"LRU":("lru",{}),"rank EMA":("ema",dict(signal="rank",alpha=0.5)),
          "current score":("score",{}),"score EMA":("ema",dict(signal="z",alpha=0.5))}
def replay(ranks,scores):
    rows=[]
    for layer in range(28):
        for label,(policy,params) in POLICIES.items():
            c=simulate(ranks[:,layer],scores[:,layer] if policy!="lru" else None,128,policy,**params)
            rows.append(dict(layer=layer,policy=label,**c,repeat_misses=c["misses"]-c["compulsory"]))
    return rows

def paired(a,b,index):
    diff=a-b
    samples=diff[index].sum(axis=1)/a[index].sum(axis=1)
    lo,hi=np.percentile(samples,[2.5,97.5])
    return dict(saved=int(diff.sum()),relative_drop=float(diff.sum()/a.sum()),ci_low=float(lo),
                ci_high=float(hi),wins=int((diff>0).sum()),ties=int((diff==0).sum()),losses=int((diff<0).sum()))
def main():
    out=Path("result")
    docs=json.loads((out/"original_manifest.json").read_text())["heldout"]
    with (out/"l4_trace_checks.csv").open() as f: checks=list(csv.DictReader(f))
    assert len(checks)==40 and [c["doc"] for c in checks]==[d["request_id"] for d in docs]
    assert all(c["passed"]=="True" for c in checks)
    totals=[]; layers=[]
    for i,d in enumerate(docs,1):
        rid=d["request_id"]
        with np.load(out/"scores"/f"{i:02d}.npz",allow_pickle=False) as z:
            assert str(z["rid"])==rid and int(z["prompt_tokens"])==d["prompt_tokens"]
            ranks, scores=z["ranked"], z["scores"]
            rows=replay(ranks,scores)
            for r in rows:
                assert r["accesses"]==len(ranks)*64
                assert r["compulsory"]==len(np.unique(ranks[:,r["layer"]]))
        layers.extend(dict(doc=rid,**r) for r in rows)
        rec=dict(doc=rid,task=rid.split("/")[0],band=d["length_band_start"])
        for name in POLICIES:
            group=[r for r in rows if r["policy"]==name]
            rec[name]=sum(r["repeat_misses"] for r in group)
        totals.append(rec)
    write_csv(out/"by_layer.csv",layers)
    write_csv(out/"by_document.csv",totals)
    counts={k:np.array([r[k] for r in totals]) for k in POLICIES}
    strata=defaultdict(list)
    for i,r in enumerate(totals): strata[r["task"],r["band"]].append(i)
    assert sorted(map(len,strata.values()))==[10]*4
    rng=np.random.default_rng(20261119)
    sample=np.concatenate([rng.choice(s,(10000,len(s))) for s in strata.values()],axis=1)
    summary=[]
    for name,b in counts.items():
        p=paired(counts["LRU"],b,sample)
        summary.append(dict(policy=name,repeat_misses=int(b.sum()),**p,
                            exceeds_heldout_10pct=10*p["saved"]>int(counts["LRU"].sum()),
                            ci_lower_exceeds_10pct=p["ci_low"]>0.10))
    write_csv(out/"summary.csv",summary)
    increment=paired(counts["current score"],counts["score EMA"],sample)
    write_csv(out/"ema_increment.csv",[dict(baseline="current score",policy="score EMA",**increment)])
    lru=int(counts["LRU"].sum())
    print(f"L4 LRU {lru:,} repeat misses. strictly over 10%: at most {(9*lru-1)//10:,}.")
    print(f"Score EMA vs current score: {increment['saved']:+,} repeat misses saved, "
          f"{increment['relative_drop']:+.2%}, paired 95% CI [{increment['ci_low']:+.2%}, {increment['ci_high']:+.2%}]")
    for r in summary:
        print(f"{r['policy']:15} {r['repeat_misses']:7,}   {r['relative_drop']:+.2%} "
              f" [95% CI {r['ci_low']:+.2%}, {r['ci_high']:+.2%}]   exceeds 10%: {r['exceeds_heldout_10pct']}")
if __name__=="__main__": main()

### 2 · 采集和回放
检查文档 ID、原文哈希、提示词长度。按每层每步的 FP32 分数稳定排序，验证能还原 L4 实际选择的 64 个页及顺序。通过后保存分数和生成 token，回放 LRU、rank EMA、当前分数、score EMA。

In [ ]:
import traceback
from datetime import datetime, timezone
out = Path("result")
if out.exists():
    raise RuntimeError("本目录已有运行记录。请从环境初始化格重新运行以创建新目录。")
out.mkdir()
def run_logged(script):
    with (out/"run.log").open("a") as log, subprocess.Popen(
            [sys.executable,"-u",script],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as p:
        for line in p.stdout:
            print(line,end="",flush=True)
            log.write(line)
        if p.wait():
            raise RuntimeError(script+" 失败，详细原因见 run.log")

state={"complete":False,"stage":"capture","experiment":"fresh_L4_short_heldout","manifest_sha256":hashlib.sha256(Path("frozen_heldout_manifest.json").read_bytes()).hexdigest()}
try:
    run_logged("capture.py")
    state["stage"]="replay"
    run_logged("compare.py")
    state.update(stage="finished",complete=True)
except Exception as e:
    state["exception"]=str(e)
    print(traceback.format_exc())
finally:
    state["utc"]=datetime.now(timezone.utc).isoformat()
    (out/"status.json").write_text(json.dumps(state,indent=2))
    for name in ("capture.py","validation.py","compare.py","sim.py"):
        shutil.copy2(name,out/name)
    result_zip=run_dir/("KV_L4_Short_Heldout_Quest_results.zip" if state["complete"] else "KV_L4_Short_Heldout_Quest_diagnostics.zip")
    # The score NPZ files are already compressed.
    from zipfile import ZIP_STORED
    with ZipFile(result_zip,"w",compression=ZIP_STORED,allowZip64=True) as result:
        for path in sorted(out.rglob("*")):
            if path.is_file(): result.write(path,path.relative_to(out))
    print("实验状态",state,"输出",result_zip)

### 3 · 下载结果
结果包包含 40 份 L4 排名与分数、逐层和逐文档计数、summary.csv、ema_increment.csv 和运行环境。脚本失败会导出诊断包，原因见 run.log。

In [ ]:
from google.colab import files
files.download(str(result_zip))